# The Standard Model in FeynPy — complete `SM.fr` comparison

This notebook is the executable Standard Model validation walkthrough for FeynPy.
It builds the packaged model, inspects the gauge-basis declaration and physical
field definitions, extracts representative Feynman rules, and proves agreement
with the tracked FeynRules `SM.fr` oracle.

Current validated result: **163/163 nonzero tree-level three- and four-point
interaction vertices match exactly after documented symbolic canonicalization.**

The reference data is tracked under `tests/fixtures/feynrules/sm/`; no local
Mathematica or ignored `sandbox/` files are required to run this notebook.
For the literate, declaration-by-declaration model implementation, see `SM_feynpy.ipynb`. This notebook is intentionally focused on validation.


## 1. Setup


In [1]:
import hashlib
import json
import platform
import re
import sys
import textwrap
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
for entry in (ROOT, SRC):
    if str(entry) not in sys.path:
        sys.path.insert(0, str(entry))

FIXTURE_DIR = ROOT / "tests" / "fixtures" / "feynrules" / "sm"

from symbolica import AtomType

from feynrules.comparison import load_feynrules_json
from theories import build_standard_model
from theories.standard_model_feynrules import (
    compare_feynrules_standard_model_vertices,
    standard_model_feynrules_field_map,
    standard_model_feynrules_name_aliases,
)

ANSI = re.compile(r"\x1B\[[0-?]*[ -/]*[@-~]")
DISPLAY_PREFIX = re.compile(r"(?:python|spenso(?:_python)?)::(?:\{[^}]*\}::)?")
DISPLAY_TAG = re.compile(r"\{[^}]*\}::")


def clean(value):
    """Render a FeynPy/Symbolica object without backend namespace noise."""
    if hasattr(value, "to_symbolica") and not hasattr(value, "to_canonical_string"):
        value = value.to_symbolica()
    rendered = (
        value.to_canonical_string()
        if hasattr(value, "to_canonical_string")
        else str(value)
    )
    rendered = ANSI.sub("", rendered)
    rendered = DISPLAY_PREFIX.sub("", rendered)
    return DISPLAY_TAG.sub("", rendered)


def print_wrapped(label, value, width=112):
    """Print one labeled symbolic expression in readable wrapped lines."""
    print(label)
    rendered = clean(value)
    readable = (
        rendered.replace("+-", " - ")
        .replace("+", " + ")
        .replace("*", " * ")
    )
    lines = textwrap.wrap(
        readable,
        width=width,
        break_long_words=False,
        break_on_hyphens=False,
    ) or [""]
    for line in lines:
        print("  " + line)


def additive_term_count(expression):
    if expression is None:
        return 0
    expanded = expression.expand()
    return len(tuple(expanded)) if expanded.get_type() == AtomType.Add else 1


def print_sector_report(label, references, row_by_key):
    """Print every aligned vertex in one sector, including exact-zero status."""
    print(f"{label} sector — {len(references)} vertices")
    header = f"{'id':>4}  {'signature':<30} {'legs':>4} {'FP terms':>8} {'FR terms':>8} {'difference':>10}"
    print(header)
    print("-" * len(header))
    for reference in references:
        row = row_by_key[reference.key]
        difference = (
            row.difference.to_canonical_string()
            if row.difference is not None
            else "n/a"
        )
        print(
            f"{str(reference.identifier):>4}  {reference.key:<30} "
            f"{len(reference.fields):>4} "
            f"{additive_term_count(row.feynpy_rule):>8} "
            f"{additive_term_count(row.feynrules_rule):>8} "
            f"{difference:>10}"
        )
    matched = sum(row_by_key[reference.key].matches for reference in references)
    print(f"\nResult: {matched}/{len(references)} MATCH")
    assert matched == len(references)


def print_vertex_comparison(row):
    print("=" * 112)
    print(f"Vertex: {row.reference.key}    fields={row.reference.fields}")
    print_wrapped("FeynPy canonical rule:", row.feynpy_rule)
    print_wrapped("FeynRules canonical rule:", row.feynrules_rule)
    print_wrapped("Canonical difference:", row.difference)


print("Repository root :", ROOT)
print("Fixture directory:", FIXTURE_DIR.relative_to(ROOT))
print("Python          :", platform.python_version())
print("Setup complete  : comparison and display helpers loaded")

Repository root : /Users/rems/Documents/MScThesis
Fixture directory: tests/fixtures/feynrules/sm
Python          : 3.14.3
Setup complete  : comparison and display helpers loaded


## 2. Tracked FeynRules oracle and provenance

The oracle was exported from FeynRules `SM.fr` version 1.4.7 in Feynman gauge
with `MinParticles -> 3`, `MaxParticles -> 4`, and
`FlavorExpand -> Generation`.

Two hashes are intentionally distinct:

- `source_sha256` identifies the original Mathematica `SM.fr` source;
- the export hash identifies the serialized 163-vertex JSON fixture.

In [2]:
model_metadata = json.loads(
    (FIXTURE_DIR / "sm_model_FeynRules.json").read_text(encoding="utf-8")
)
full_references = load_feynrules_json(
    FIXTURE_DIR / "sm_vertices_FeynRules.json"
)
sector_references = {
    sector: load_feynrules_json(
        FIXTURE_DIR / f"{sector}_vertices_FeynRules.json"
    )
    for sector in ("gauge", "matter", "yukawa", "higgs", "ghost")
}

source_hash = model_metadata["source_sha256"]
export_hash = hashlib.sha256(
    (FIXTURE_DIR / "sm_vertices_FeynRules.json").read_bytes()
).hexdigest()
version_match = re.search(r'Version -> "([^"]+)"', model_metadata["information"])

print("FeynRules model       :", model_metadata["model_name"])
print("SM.fr version         :", version_match.group(1) if version_match else "unknown")
print("Feynman gauge         :", model_metadata["feynman_gauge"])
print("SM.fr source SHA-256  :", source_hash)
print("vertex JSON SHA-256   :", export_hash)
print("full interaction count:", len(full_references))
print()
print(f"{'sector':<10} {'vertices':>8}")
print("-" * 19)
for sector, references in sector_references.items():
    print(f"{sector:<10} {len(references):>8}")

assert source_hash == "44690e769ecc4ed649033d2f9d58c5672203d8e820c56c90a378464204c99edc"
assert export_hash == "01dc1b98feb6a112b65e8c1cb42aa9e005fb6ae362767850d7b5888e8689913f"
assert len(full_references) == 163
assert sum(map(len, sector_references.values())) == 163

full_payload = {
    reference.key: (reference.fields, reference.legs, reference.rule)
    for reference in full_references
}
sector_payload = {
    reference.key: (reference.fields, reference.legs, reference.rule)
    for references in sector_references.values()
    for reference in references
}
assert sector_payload == full_payload
print("\nProvenance checks     : PASS")
print("Sector partition      : PASS (exact fields, legs, and rules)")

FeynRules model       : Standard Model
SM.fr version         : 1.4.7
Feynman gauge         : True
SM.fr source SHA-256  : 44690e769ecc4ed649033d2f9d58c5672203d8e820c56c90a378464204c99edc
vertex JSON SHA-256   : 01dc1b98feb6a112b65e8c1cb42aa9e005fb6ae362767850d7b5888e8689913f
full interaction count: 163

sector     vertices
-------------------
gauge             8
matter           51
yukawa           42
higgs            38
ghost            24

Provenance checks     : PASS
Sector partition      : PASS (exact fields, legs, and rules)


## 3. Build the packaged Standard Model

`build_standard_model()` is the single authoritative implementation used by the
test suite. It declares the unbroken gauge-basis model, compiles covariant
derivatives and field strengths, unfolds finite weak indices, and applies all
field definitions in one simultaneous transformation stage.

In [3]:
sm = build_standard_model(
    include_ghosts=True,
    include_gauge_fixing=False,
)
source_validation = sm.source_model.validate()
physical_validation = sm.model.validate()
declared_physical_fields = (
    sm.fields.vl, sm.fields.l, sm.fields.uq, sm.fields.dq,
    sm.fields.H, sm.fields.G0, sm.fields.GP, sm.fields.W,
    sm.fields.Z, sm.fields.A, sm.fields.G, sm.fields.ghA,
    sm.fields.ghZ, sm.fields.ghWp, sm.fields.ghWm, sm.fields.ghG,
)

unexpanded_signatures = sm.lagrangian.vertex_signatures()
expanded_signatures = sm.lagrangian.vertex_signatures(flavor_expand=True)
arity_counts = Counter(signature.arity for signature in expanded_signatures)

print("Model name                  :", sm.model.name)
print("Source gauge groups         :", len(sm.source_model.gauge_groups))
print("Source field classes        :", len(sm.source_model.fields))
print("Declared physical classes    :", len(declared_physical_fields))
print("Registered fields + members  :", len(sm.model.fields))
print("Compiled interaction terms  :", len(sm.lagrangian.terms))
print("Unexpanded signatures       :", len(unexpanded_signatures))
print("Flavor-expanded signatures  :", len(expanded_signatures))
print("Expanded arity distribution :", dict(sorted(arity_counts.items())))
print("Source-model validation     :", "PASS" if source_validation.ok else source_validation)
print("Physical-model validation   :", "PASS" if physical_validation.ok else physical_validation)

assert source_validation.ok
assert physical_validation.ok
assert len(declared_physical_fields) == 16
assert arity_counts == {2: 30, 3: 139, 4: 32}

Model name                  : Standard Model
Source gauge groups         : 3
Source field classes        : 12
Declared physical classes    : 16
Registered fields + members  : 28
Compiled interaction terms  : 306
Unexpanded signatures       : 123
Flavor-expanded signatures  : 201
Expanded arity distribution : {2: 30, 3: 139, 4: 32}
Source-model validation     : PASS
Physical-model validation   : PASS


## 4. Gauge groups, physical fields, and parameters



In [4]:
print("Gauge groups")
header = f"{'name':<8} {'type':<12} {'coupling':<10} {'gauge field':<12} {'charge/representation'}"
print(header)
print("-" * len(header))
for public_name, group in sm.gauge_groups.__dict__.items():
    representation = group.charge if group.abelian else ", ".join(
        item.name for item in group.representations
    )
    print(
        f"{public_name:<8} "
        f"{('abelian' if group.abelian else 'non-abelian'):<12} "
        f"{clean(group.coupling.symbol):<10} "
        f"{group.gauge_boson.name:<12} "
        f"{representation}"
    )

print("\nRegistered physical fields (declared classes and flavor members)")
header = f"{'field':<8} {'kind':<8} {'spin':<6} {'self-conj':<10} {'indices':<28} {'members'}"
print(header)
print("-" * len(header))
for field in sm.model.fields:
    indices = ",".join(index.name for index in field.indices) or "-"
    members = ",".join(member.name for member in field.class_members) or "-"
    print(
        f"{field.name:<8} {field.kind:<8} {str(field.spin):<6} "
        f"{str(field.self_conjugate):<10} {indices:<28} {members}"
    )

Gauge groups
name     type         coupling   gauge field  charge/representation
-------------------------------------------------------------------
U1Y      abelian      g1         B            Y
SU2L     non-abelian  gw         Wi           doublet
SU3C     non-abelian  gs         G            fundamental

Registered physical fields (declared classes and flavor members)
field    kind     spin   self-conj  indices                      members
------------------------------------------------------------------------
vl       fermion  1/2    False      Spinor,Generation            ve,vm,vt
l        fermion  1/2    False      Spinor,Generation            e,mu,ta
uq       fermion  1/2    False      Spinor,Generation,ColorFund  u,c,t
dq       fermion  1/2    False      Spinor,Generation,ColorFund  d,s,b
H        scalar   0      True       -                            -
G0       scalar   0      True       -                            -
GP       scalar   0      False      -                   

In [5]:
print("Standard Model parameters used by the packaged builder")
header = f"{'attribute':<10} {'symbol':<12} {'rank':>4} {'type':<10} {'value/definition'}"
print(header)
print("-" * len(header))
for attribute, parameter in sm.parameters.__dict__.items():
    value = clean(parameter.value) if parameter.value is not None else "symbolic"
    parameter_type = "internal" if parameter.internal else "external"
    print(
        f"{attribute:<10} {clean(parameter.symbol):<12} "
        f"{len(parameter.indices):>4} {parameter_type:<10} {value}"
    )

Standard Model parameters used by the packaged builder
attribute  symbol       rank type       value/definition
--------------------------------------------------------
g1         g1              0 internal   symbolic
g2         gw              0 internal   symbolic
g3         gs              0 internal   symbolic
lam        lam             0 internal   symbolic
vev        vev             0 internal   symbolic
Mvl        Mvl             1 internal   symbolic
Ml         Ml              1 internal   symbolic
Mu         Mu              1 internal   symbolic
Md         Md              1 internal   symbolic
MW         MW              0 internal   1/2*gw*vev
MZ         MZ              0 internal   (g1^2+gw^2)^(1/2)*1/2*vev
MH         MH              0 internal   (2*lam*vev^2)^(1/2)
sw         sw              0 internal   symbolic
cw         cw              0 internal   symbolic
ee         ee              0 internal   symbolic
xiA        xiA             0 external   1
xiZ        xiZ          

## 5. Gauge-basis Lagrangian and physical-field definitions

The source declaration mirrors the `SM.fr` split
`LGauge + LFermions + LHiggs + LYukawa + LGhost`. The comparison build does not
add a separate gauge-fixing Lagrangian because the exported oracle contains only
three- and four-point interactions; Feynman-gauge ghost and Goldstone
interactions are included.

In [6]:
for sector_name in (
    "LGauge",
    "LFermions",
    "LHiggs",
    "LYukawa",
    "LGaugeFixing",
    "LGhost",
):
    declaration = getattr(sm.lagrangians, sector_name)
    print("=" * 112)
    print(f"{sector_name}: {len(declaration.source_terms)} declared source term(s)")
    print_wrapped("Declaration:", str(declaration))

print("\nAll source sectors displayed.")

LGauge: 3 declared source term(s)
Declaration:
  -1/4  *  FieldStrength(U1Y, mu, nu)  *  FieldStrength(U1Y, mu, nu)  +  -1/4  *  FieldStrength(SU2L, mu, nu, aW)
  *  FieldStrength(SU2L, mu, nu, aW)  +  -1/4  *  FieldStrength(SU3C, mu, nu, aC)  *  FieldStrength(SU3C, mu, nu,
  aC)
LFermions: 5 declared source term(s)
Declaration:
  1𝑖  *  QL.bar  *  Gamma(mu)  *  CovD(QL, mu)  +  1𝑖  *  LL.bar  *  Gamma(mu)  *  CovD(LL, mu)  +  1𝑖  *  uR.bar
  *  Gamma(mu)  *  CovD(uR, mu)  +  1𝑖  *  dR.bar  *  Gamma(mu)  *  CovD(dR, mu)  +  1𝑖  *  lR.bar  *  Gamma(mu)
  *  CovD(lR, mu)
LHiggs: 3 declared source term(s)
Declaration:
  CovD(Phi.bar, mu)  *  CovD(Phi, mu)  +  lam * vev^2  *  Phi.bar  *  Phi  +  -lam  *  Phi.bar  *  Phi  *  Phi.bar
  *  Phi
LYukawa: 6 declared source term(s)
Declaration:
  -yd(ff2,ff3) * CKM(ff1,ff2)  *  QL.bar  *  dR  *  Phi  +  -yl(ff1,ff3)  *  LL.bar  *  lR  *  Phi  +
  -weak_eps2(cof(2, ii),cof(2, jj)) * yu(ff1,ff2)  *  QL.bar  *  uR  *  Phi.bar  +  -YdDag(ff3,ff2) *
 

In [7]:
print("Gauge basis -> physical basis definitions")
header = f"{'source':<8} {'fixed components':<20} {'replacement'}"
print(header)
print("-" * len(header))
for transformation in sm.transformations:
    components = str(dict(transformation.components)) if transformation.components else "all"
    print(
        f"{transformation.source.name:<8} "
        f"{components:<20} "
        f"{clean(transformation.expr)}"
    )
print(f"\nTotal definitions: {len(sm.transformations)}")

Gauge basis -> physical basis definitions
source   fixed components     replacement
-----------------------------------------
B        all                  -sw * Z + cw * A
Wi       {1: 1}               (1/2)^(1/2) * W.bar + (1/2)^(1/2) * W
Wi       {1: 2}               -1𝑖*(1/2)^(1/2) * W.bar + 1𝑖*(1/2)^(1/2) * W
Wi       {1: 3}               cw * Z + sw * A
Phi      {0: 1}               -1𝑖 * GP
Phi      {0: 2}               (1/2)^(1/2)*vev + (1/2)^(1/2) * H + 1𝑖*(1/2)^(1/2) * G0
LL       {1: 1}               ProjM * vl
LL       {1: 2}               ProjM * l
lR       all                  ProjP * l
QL       {1: 1}               ProjM * uq
QL       {1: 2}               CKM * ProjM * dq
uR       all                  ProjP * uq
dR       all                  ProjP * dq
ghB      all                  -sw * ghZ + cw * ghA
ghWi     {0: 1}               (1/2)^(1/2) * ghWp + (1/2)^(1/2) * ghWm
ghWi     {0: 2}               1𝑖*(1/2)^(1/2) * ghWp + -1𝑖*(1/2)^(1/2) * ghWm
ghWi     {0: 3}         

## 6. Representative FeynPy rules

These are direct outputs of the physical compiled Lagrangian. External
wavefunctions and the universal momentum-conservation delta are omitted, which
is also the convention used by the FeynRules JSON export.

In [8]:
field_map = standard_model_feynrules_field_map(sm.fields)
name_aliases = standard_model_feynrules_name_aliases(sm.fields)

examples = (
    ("W- W+ photon", ("A", "W", "Wbar")),
    ("Higgs W- W+", ("H", "W", "Wbar")),
    ("electron QED current", ("ebar", "e", "A")),
    ("CKM charged current", ("tbar", "b", "W")),
    ("top-Higgs Yukawa", ("tbar", "t", "H")),
    ("ghost-gluon", ("ghGbar", "ghG", "G")),
)

for label, names in examples:
    rule = sm.lagrangian.feynman_rule(
        *(field_map[name] for name in names),
        simplify=True,
        include_delta=False,
        flavor_expand=True,
    )
    print("=" * 112)
    print(f"{label}: {names}")
    print_wrapped("FeynPy rule:", rule)

print("\nRepresentative rule extraction: PASS")

W- W+ photon: ('A', 'W', 'Wbar')
FeynPy rule:
  -1𝑖 * ee * pcomp(q1,mu3) * g(mink(4,mu1),mink(4,mu2)) - 1𝑖 * ee * pcomp(q2,mu1) * g(mink(4,mu2),mink(4,mu3)) -
  1𝑖 * ee * pcomp(q3,mu2) * g(mink(4,mu1),mink(4,mu3)) + 1𝑖 * ee * pcomp(q1,mu2) * g(mink(4,mu1),mink(4,mu3)) + 1𝑖
  * ee * pcomp(q2,mu3) * g(mink(4,mu1),mink(4,mu2)) + 1𝑖 * ee * pcomp(q3,mu1) * g(mink(4,mu2),mink(4,mu3))
Higgs W- W+: ('H', 'W', 'Wbar')
FeynPy rule:
  1𝑖/2 * ee^2 * sw^(-2) * vev * g(mink(4,mu2),mink(4,mu3))
electron QED current: ('ebar', 'e', 'A')
FeynPy rule:
  -1𝑖/2 * ee * gamma(bis(4,i1),bis(4,i2),mink(4,mu3)) + 1𝑖/2 * ee *
  gamma(bis(4,bis_canon_1),bis(4,bis_canon_2),mink(4,mu3)) * gamma5(bis(4,bis_canon_2),bis(4,i2)) *
  gamma5(bis(4,i1),bis(4,bis_canon_1))
CKM charged current: ('tbar', 'b', 'W')
FeynPy rule:
  (1/2)^(1/2) * -1𝑖/4 * CKM33 * ee * sw^(-1) * g(cof(3,c1),cof(3,c2)) *
  gamma(bis(4,bis_canon_1),bis(4,bis_canon_2),mink(4,mu3)) * gamma5(bis(4,bis_canon_2),bis(4,i2)) *
  gamma5(bis(4,i1),bis(4,bis_

### 6.1 Inspecting the validated compiled model

The general inspection workflow from the former capabilities walkthrough is
retained here. `vertex_report` summarizes coverage, `matching_terms` shows
the compiled monomials behind a vertex, and flavor expansion exposes member
fields without rebuilding the model.


In [9]:
report = sm.lagrangian.vertex_report()
hww_terms = sm.lagrangian.matching_terms(
    sm.fields.H, sm.fields.W.bar, sm.fields.W
)
wwa_terms = sm.lagrangian.matching_terms(
    sm.fields.W.bar, sm.fields.W, sm.fields.A
)
electron = sm.fields.l.class_members[0]
electron_rule = sm.lagrangian.feynman_rule(
    electron.bar, electron, sm.fields.A,
    flavor_expand=True,
    simplify=True,
)

print("vertex report terms       :", report.total_terms)
print("vertex report signatures  :", report.total_signatures)
print("H W- W+ contributing terms:", len(hww_terms))
print("W- W+ A contributing terms:", len(wwa_terms))
print("signatures containing W-/W+:")
for signature in sm.lagrangian.vertex_signatures(
    contains_fields=(sm.fields.W.bar, sm.fields.W)
)[:10]:
    print(" ", signature.names, "terms=", signature.term_count)
print()
print_wrapped("Flavor-expanded electron-photon rule:", electron_rule)


vertex report terms       : 306
vertex report signatures  : 123
H W- W+ contributing terms: 1
W- W+ A contributing terms: 24
signatures containing W-/W+:
  ('W.bar', 'W') terms= 5
  ('H', 'W.bar', 'W') terms= 1
  ('W.bar', 'W', 'A') terms= 24
  ('W.bar', 'W', 'Z') terms= 24
  ('G0', 'G0', 'W.bar', 'W') terms= 1
  ('GP.bar', 'GP', 'W', 'W.bar') terms= 1
  ('H', 'G0', 'W.bar', 'W') terms= 2
  ('H', 'H', 'W.bar', 'W') terms= 1
  ('W.bar', 'A', 'W', 'A') terms= 8
  ('W.bar', 'W', 'W.bar', 'W') terms= 4

Flavor-expanded electron-photon rule:
  -1𝑖/2 * ee * gamma(bis(4,i1),bis(4,i2),mink(4,mu3)) + 1𝑖/2 * ee *
  gamma(bis(4,bis_canon_1),bis(4,bis_canon_2),mink(4,mu3)) * gamma5(bis(4,bis_canon_2),bis(4,i2)) *
  gamma5(bis(4,i1),bis(4,bis_canon_1))


## 7. Exact symbolic comparison method

The comparison is not a text diff. It:

1. aligns field multisets but preserves the original FeynRules external-leg order;
2. parses `ME`, `FV`, `Ga`, `T`, `f`, projectors, deltas, Yukawas, and CKM entries
   into native Symbolica/Spenso tensors;
3. canonicalizes dummy indices, color tensors, structure constants, and spinor
   chains independently on both sides;
4. applies all-incoming momentum conservation where required for ghost rules;
5. declares `MATCH` only when the exact symbolic difference is zero.

The next cell performs the complete comparison once. A raw-output subsection
then shows two untouched examples, followed by five tables containing **every
one of the 163 aligned vertices**.

In [10]:
full_comparison = compare_feynrules_standard_model_vertices(
    sm.lagrangian,
    full_references,
    field_map=field_map,
    feynpy_name_aliases=name_aliases,
)
row_by_key = {row.reference.key: row for row in full_comparison.rows}

print("FeynRules vertices :", len(full_references))
print("FeynPy matches     :", full_comparison.matched)
print("FeynRules-only     :", full_comparison.feynrules_only)
print("FeynPy-only        :", full_comparison.feynpy_only)
print("Complete agreement :", full_comparison.all_match)

assert full_comparison.matched == 163
assert full_comparison.all_match

FeynRules vertices : 163
FeynPy matches     : 163
FeynRules-only     : ()
FeynPy-only        : ()
Complete agreement : True


### 7.1 Original FeynRules and FeynPy outputs — no normalization

This section deliberately shows the two engines **before** the comparison
adapter translates tensor heads, renames fields or indices, reduces projectors,
or applies momentum identities.

- The FeynRules text is printed exactly as stored in the JSON oracle.
- The FeynPy text is the direct native `to_canonical_string()` output.
- No attempt is made here to make the spellings look alike.

Two examples are useful:

1. `H|H|H` is small and already visibly identical apart from syntax for the
   imaginary unit and backend-qualified symbol names.
2. `G|ghG|ghGbar` is intentionally harder: the derivative momentum and color
   index order look different in the original outputs.

In [11]:
def original_outputs(key):
    """Return the untouched FeynRules string and direct native FeynPy string."""
    reference = next(item for item in full_references if item.key == key)
    feynpy_rule = sm.lagrangian.feynman_rule(
        *(field_map[name] for name in reference.fields),
        simplify=True,
        include_delta=False,
        flavor_expand=True,
    )
    return reference, reference.rule, feynpy_rule.to_canonical_string()


for key, description in (
    ("H|H|H", "Small example: cubic Higgs self-interaction"),
    ("G|ghG|ghGbar", "Nontrivial example: ghost-gluon interaction"),
):
    reference, feynrules_original, feynpy_original = original_outputs(key)
    final_row = row_by_key[key]
    print("=" * 112)
    print(description)
    print("Signature / leg order:", reference.fields)
    print("\nOriginal FeynRules output (unchanged JSON string):")
    print(feynrules_original)
    print("\nOriginal FeynPy output (unchanged native canonical string):")
    print(feynpy_original)
    print("\nRaw strings identical?:", feynrules_original == feynpy_original)
    print("Final symbolic comparison status:", final_row.status)
    print("Final canonical difference:", final_row.difference.to_canonical_string())

assert row_by_key["H|H|H"].matches
assert row_by_key["G|ghG|ghGbar"].matches

Small example: cubic Higgs self-interaction
Signature / leg order: ('H', 'H', 'H')

Original FeynRules output (unchanged JSON string):
(-6*I)*lam*vev

Original FeynPy output (unchanged native canonical string):
-6𝑖*python::{}::lam*python::{}::vev

Raw strings identical?: False
Final symbolic comparison status: MATCH
Final canonical difference: 0
Nontrivial example: ghost-gluon interaction
Signature / leg order: ('ghGbar', 'ghG', 'G')

Original FeynRules output (unchanged JSON string):
gs*f[Index[Gluon, Ext[3]], Index[Gluon, Ext[1]], Index[Gluon, Ext[2]]]*FV[2, Index[Lorentz, Ext[3]]] + gs*f[Index[Gluon, Ext[3]], Index[Gluon, Ext[1]], Index[Gluon, Ext[2]]]*FV[3, Index[Lorentz, Ext[3]]]

Original FeynPy output (unchanged native canonical string):
python::{}::gs*python::{}::pcomp(python::{}::q1,python::{}::mu3)*spenso::{real}::f(spenso::{spenso::upper}::coad(8,python::{}::a1),spenso::{spenso::upper}::coad(8,python::{}::a3),spenso::{spenso::upper}::coad(8,python::{}::a2))

Raw strings iden

#### cubic Higgs vertex

$$
\text{FeynRules}:\quad (-6 I)\lambda v
\qquad
\text{FeynPy}:\quad -6i\lambda v.
$$

#### ghost--gluon 
$$
\text{FeynRules}:\quad
g_s f^{a_3 a_1 a_2}(q_2+q_3)_{\mu_3},
$$

$$
\text{FeynPy}:\quad
g_s f^{a_1 a_3 a_2}(q_1)_{\mu_3}.
$$

They are equal in two short steps:

1. Antisymmetry of the structure constant gives
   $f^{a_3 a_1 a_2}=-f^{a_1 a_3 a_2}$.
2. All momenta are incoming, so $q_1+q_2+q_3=0$, hence
   $q_2+q_3=-q_1$.



### 7.2 Pure-gauge sector — all 8 outputs

In [12]:
print_sector_report(
    "Pure gauge",
    sector_references["gauge"],
    row_by_key,
)

Pure gauge sector — 8 vertices
  id  signature                      legs FP terms FR terms difference
----------------------------------------------------------------------
   4  A|A|W|Wbar                        4        3        3          0
   3  A|W|Wbar                          3        6        6          0
   7  A|W|Wbar|Z                        4        3        3          0
   1  G|G|G                             3        6        6          0
   2  G|G|G|G                           4        6        6          0
   5  W|Wbar|Z                          3        6        6          0
   8  W|Wbar|Z|Z                        4        3        3          0
   6  W|W|Wbar|Wbar                     4        3        3          0

Result: 8/8 MATCH


### 7.3 Fermion–gauge matter sector — all 51 outputs

In [13]:
print_sector_report(
    "Fermion-gauge matter",
    sector_references["matter"],
    row_by_key,
)

Fermion-gauge matter sector — 51 vertices
  id  signature                      legs FP terms FR terms difference
----------------------------------------------------------------------
   7  A|b|bbar                          3        1        1          0
   4  A|c|cbar                          3        1        1          0
   8  A|d|dbar                          3        1        1          0
   1  A|e|ebar                          3        1        1          0
   2  A|mu|mubar                        3        1        1          0
   9  A|s|sbar                          3        1        1          0
   3  A|ta|tabar                        3        1        1          0
   5  A|t|tbar                          3        1        1          0
   6  A|u|ubar                          3        1        1          0
  25  bbar|c|Wbar                       3        2        2          0
  26  bbar|t|Wbar                       3        2        2          0
  27  bbar|u|Wbar                  

### 7.4 Yukawa and fermion–Goldstone sector — all 42 outputs

In [14]:
print_sector_report(
    "Yukawa / fermion-Goldstone",
    sector_references["yukawa"],
    row_by_key,
)

Yukawa / fermion-Goldstone sector — 42 vertices
  id  signature                      legs FP terms FR terms difference
----------------------------------------------------------------------
   1  bbar|c|GPbar                      3        4        4          0
   2  bbar|GPbar|t                      3        4        4          0
   3  bbar|GPbar|u                      3        4        4          0
  10  b|bbar|G0                         3        1        1          0
  13  b|bbar|H                          3        1        1          0
  25  b|cbar|GP                         3        4        4          0
  28  b|GP|tbar                         3        4        4          0
  31  b|GP|ubar                         3        4        4          0
  26  cbar|d|GP                         3        4        4          0
  27  cbar|GP|s                         3        4        4          0
  34  c|cbar|G0                         3        1        1          0
  37  c|cbar|H               

### 7.5 Higgs, Goldstone, and bosonic matter sector — all 38 outputs

In [15]:
print_sector_report(
    "Higgs / Goldstone / bosonic matter",
    sector_references["higgs"],
    row_by_key,
)

Higgs / Goldstone / bosonic matter sector — 38 vertices
  id  signature                      legs FP terms FR terms difference
----------------------------------------------------------------------
  14  A|GPbar|W                         3        1        1          0
  11  A|GP|GPbar                        3        2        2          0
  19  A|GP|Wbar                         3        1        1          0
   7  G0|G0|H                           3        1        1          0
  15  G0|GPbar|W                        3        2        2          0
  20  G0|GP|Wbar                        3        2        2          0
  27  G0|H|Z                            3        4        4          0
  16  GPbar|H|W                         3        2        2          0
  31  GPbar|W|Z                         3        1        1          0
   8  GP|GPbar|H                        3        1        1          0
  28  GP|GPbar|Z                        3        4        4          0
  21  GP|H|Wbar      

### 7.6 Ghost sector — all 24 outputs

In [16]:
print_sector_report(
    "Ghost",
    sector_references["ghost"],
    row_by_key,
)

Ghost sector — 24 vertices
  id  signature                      legs FP terms FR terms difference
----------------------------------------------------------------------
   7  A|ghWm|ghWmbar                    3        1        2          0
  15  A|ghWp|ghWpbar                    3        1        2          0
   5  G0|ghWm|ghWmbar                   3        1        1          0
  13  G0|ghWp|ghWpbar                   3        1        1          0
  24  G|ghG|ghGbar                      3        1        2          0
   1  ghAbar|ghWm|W                     3        1        2          0
   2  ghAbar|ghWp|Wbar                  3        1        2          0
   3  ghA|ghWmbar|GPbar                 3        1        1          0
   4  ghA|ghWmbar|Wbar                  3        1        2          0
  11  ghA|ghWpbar|GP                    3        1        1          0
  12  ghA|ghWpbar|W                     3        1        2          0
   9  ghWmbar|ghZ|GPbar                 3        2

## 8. Complete 163/163 summary

The sector totals below are computed from the actual reports above, not copied
from markdown text.

In [17]:
print(f"{'sector':<34} {'matched':>8} {'total':>8}")
print("-" * 52)
total_matched = 0
total_vertices = 0
for sector, label in (
    ("gauge", "pure gauge"),
    ("matter", "fermion-gauge matter"),
    ("yukawa", "Yukawa / fermion-Goldstone"),
    ("higgs", "Higgs / Goldstone / bosonic"),
    ("ghost", "ghost"),
):
    references = sector_references[sector]
    matched = sum(row_by_key[reference.key].matches for reference in references)
    total_matched += matched
    total_vertices += len(references)
    print(f"{label:<34} {matched:>8} {len(references):>8}")
print("-" * 52)
print(f"{'TOTAL':<34} {total_matched:>8} {total_vertices:>8}")
print("\nExact symbolic result: 163/163 MATCH")

assert (total_matched, total_vertices) == (163, 163)

sector                              matched    total
----------------------------------------------------
pure gauge                                8        8
fermion-gauge matter                     51       51
Yukawa / fermion-Goldstone               42       42
Higgs / Goldstone / bosonic              38       38
ghost                                    24       24
----------------------------------------------------
TOTAL                                   163      163

Exact symbolic result: 163/163 MATCH


## 9. Global nonzero-signature coverage

Sector adapters deliberately filter candidate names to their reference sector.
This independent check evaluates every flavor-expanded FeynPy candidate of
arity three or four. Eight candidate field multisets cancel to the exact zero
rule; the remaining nonzero set must equal the full 163-entry FeynRules set.

In [18]:
reference_signatures = {reference.signature for reference in full_references}
nonzero_signatures = set()
zero_candidates = set()

for signature in sm.lagrangian.vertex_signatures(flavor_expand=True):
    if signature.arity not in (3, 4):
        continue
    normalized = tuple(
        sorted(name_aliases.get(name, name) for name in signature.names)
    )
    rule = sm.lagrangian.feynman_rule(
        *signature.fields,
        simplify=True,
        include_delta=False,
        flavor_expand=True,
    )
    if rule.to_canonical_string() == "0":
        zero_candidates.add(normalized)
    else:
        nonzero_signatures.add(normalized)

print("Candidate arity-3/4 signatures:", len(nonzero_signatures) + len(zero_candidates))
print("Exact-zero cancellations        :", len(zero_candidates))
for signature in sorted(zero_candidates):
    print("  zero:", signature)
print("Nonzero FeynPy signatures      :", len(nonzero_signatures))
print("FeynRules reference signatures :", len(reference_signatures))
print("FeynRules-only                  :", sorted(reference_signatures - nonzero_signatures))
print("FeynPy-only                     :", sorted(nonzero_signatures - reference_signatures))

assert len(zero_candidates) == 8
assert nonzero_signatures == reference_signatures
print("\nGlobal nonzero-signature coverage: PASS")

Candidate arity-3/4 signatures: 171
Exact-zero cancellations        : 8
  zero: ('G0', 'G0', 'G0', 'H')
  zero: ('G0', 'G0', 'Z')
  zero: ('G0', 'GP', 'GPbar', 'H')
  zero: ('G0', 'H', 'H')
  zero: ('G0', 'H', 'H', 'H')
  zero: ('G0', 'H', 'W', 'Wbar')
  zero: ('G0', 'H', 'Z', 'Z')
  zero: ('H', 'H', 'Z')
Nonzero FeynPy signatures      : 163
FeynRules reference signatures : 163
FeynRules-only                  : []
FeynPy-only                     : []

Global nonzero-signature coverage: PASS


## 10. Representative canonical side-by-side outputs

The tables above establish coverage. These examples expose the actual canonical
expressions from both engines and their zero differences for the hardest or
most convention-sensitive structures.

In [19]:
for key in (
    "G|G|G|G",
    "b|tbar|W",
    "H|t|tbar",
    "H|Z|Z",
    "G|ghG|ghGbar",
):
    print_vertex_comparison(row_by_key[key])

print("\nRepresentative canonical comparisons: PASS")

Vertex: G|G|G|G    fields=('G', 'G', 'G', 'G')
FeynPy canonical rule:
  -1𝑖 * gs^2 * f(coad(8,a1),coad(8,a2),coad(8,canon_dummy_1_1)) * f(coad(8,a3),coad(8,a4),coad(8,canon_dummy_1_1))
  * g(mink(4,mu1),mink(4,mu3)) * g(mink(4,mu2),mink(4,mu4)) - 1𝑖 * gs^2 *
  f(coad(8,a1),coad(8,a3),coad(8,canon_dummy_1_1)) * f(coad(8,a2),coad(8,a4),coad(8,canon_dummy_1_1)) *
  g(mink(4,mu1),mink(4,mu2)) * g(mink(4,mu3),mink(4,mu4)) - 1𝑖 * gs^2 *
  f(coad(8,a1),coad(8,a4),coad(8,canon_dummy_1_1)) * f(coad(8,a2),coad(8,a3),coad(8,canon_dummy_1_1)) *
  g(mink(4,mu1),mink(4,mu2)) * g(mink(4,mu3),mink(4,mu4)) + 1𝑖 * gs^2 *
  f(coad(8,a1),coad(8,a2),coad(8,canon_dummy_1_1)) * f(coad(8,a3),coad(8,a4),coad(8,canon_dummy_1_1)) *
  g(mink(4,mu1),mink(4,mu4)) * g(mink(4,mu2),mink(4,mu3)) + 1𝑖 * gs^2 *
  f(coad(8,a1),coad(8,a3),coad(8,canon_dummy_1_1)) * f(coad(8,a2),coad(8,a4),coad(8,canon_dummy_1_1)) *
  g(mink(4,mu1),mink(4,mu4)) * g(mink(4,mu2),mink(4,mu3)) + 1𝑖 * gs^2 *
  f(coad(8,a1),coad(8,a4),coad(8,cano

## 11. Reusable rule lookup

The helper below accepts the external FeynRules field spellings used in the
fixture (`Wbar`, `tbar`, `ghGbar`, and so on) and returns the corresponding
FeynPy rule. This is the shortest reusable entry point after building `sm`.

In [20]:
def rule_for(*field_names):
    """Return a simplified flavor-expanded physical FeynPy rule."""
    return sm.lagrangian.feynman_rule(
        *(field_map[name] for name in field_names),
        simplify=True,
        include_delta=False,
        flavor_expand=True,
    )

example_rule = rule_for("tbar", "b", "W")
print("Usage example: rule_for('tbar', 'b', 'W')")
print_wrapped("Result:", example_rule)
print("\nReusable lookup helper: ready")

Usage example: rule_for('tbar', 'b', 'W')
Result:
  (1/2)^(1/2) * -1𝑖/4 * CKM33 * ee * sw^(-1) * g(cof(3,c1),cof(3,c2)) *
  gamma(bis(4,bis_canon_1),bis(4,bis_canon_2),mink(4,mu3)) * gamma5(bis(4,bis_canon_2),bis(4,i2)) *
  gamma5(bis(4,i1),bis(4,bis_canon_1)) + (1/2)^(1/2) * -1𝑖/4 * CKM33 * ee * sw^(-1) * g(cof(3,c1),cof(3,c2)) *
  gamma(bis(4,i1),bis(4,bis_canon_2),mink(4,mu3)) * gamma5(bis(4,bis_canon_2),bis(4,i2)) + (1/2)^(1/2) * 1𝑖/4 *
  CKM33 * ee * sw^(-1) * g(cof(3,c1),cof(3,c2)) * gamma(bis(4,bis_canon_1),bis(4,i2),mink(4,mu3)) *
  gamma5(bis(4,i1),bis(4,bis_canon_1)) + (1/2)^(1/2) * 1𝑖/4 * CKM33 * ee * sw^(-1) * g(cof(3,c1),cof(3,c2)) *
  gamma(bis(4,i1),bis(4,i2),mink(4,mu3))

Reusable lookup helper: ready


## Final result and precise claim

- **Gauge:** 8/8
- **Fermion–gauge matter:** 51/51
- **Yukawa and fermion–Goldstone:** 42/42
- **Higgs/Goldstone/bosonic matter:** 38/38
- **Ghost:** 24/24
- **Total:** **163/163 exact symbolic matches**

The equality is semantic, not byte-for-byte: field aliases, native tensor-head
translation, dummy-index canonicalization, chiral-basis reduction, momentum
conservation for derivative ghost rules, and `cw² + sw² = 1` where required are
part of the proof. The complete scope and remaining limitations are documented
in `docs/SM_FR_COMPARISON_REVIEW.md`.